# 同一条件风险下的基准核心与杠杆/现金表示

本批固定两个风险源：已封存repeat1的learned波动预测与close20观察方差；各自比较core与lever_cash，另设SOXX，共10条成本独立原生路径。零模型训练、零风险校准拟合，不读取均值作目标，不调用凸优化器。

参数仅来自[plan.json](plan.json)。旧源六对象使用显式 `artifact_manifest.json` 与预登记SHA，在反序列化前核验。开发期风险输入须精确复现；2023以后单独从raw风险与冻结年度校准重建，不能冒用旧诊断选择。

In [1]:
from pathlib import Path
import os,sys,json
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
root=Path.cwd().resolve()
while not (root/"AGENTS.md").is_file():
    if root==root.parent: raise RuntimeError("workspace root unavailable")
    root=root.parent
sys.path.insert(0,str(root))
import qlib
from qlib.data import D
from qlib.workflow import R
from IPython.display import display,Markdown
card=root/"research/benchmark_core_variance_20261009"
plan=json.loads((card/"plan.json").read_text())
qlib.init(provider_uri=str(root/plan["source"]["provider_uri"]),region="us",kernels=1)
calendar=D.calendar(freq="day",future=False)
print({"provider_uri":plan["source"]["provider_uri"],"as_of":plan["source"]["as_of"],
       "calendar_start":str(calendar[0]),"calendar_end":str(calendar[-1]),"fits":0,"calibration_fits":0})

[76858:MainThread](2026-10-09 14:50:28,996) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[76858:MainThread](2026-10-09 14:50:28,998) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[76858:MainThread](2026-10-09 14:50:28,998) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


{'provider_uri': 'data/qlib/published_futu_affine_20260925_v1', 'as_of': '2026-09-25', 'calendar_start': '2010-03-11 00:00:00', 'calendar_end': '2026-09-25 00:00:00', 'fits': 0, 'calibration_fits': 0}


## 固定风险映射与独立记账

两种表示使用同一个当日Sigma与年度B。core先保留SOXX核心，必要时转现金；预算高于SOXX风险时求完整二次风险曲线的最大可行SOXL比例。lever_cash仅用SOXL/现金。矩阵不加入经济地板、不假定精确3倍beta。

目标用精确有理数验证风险可行性；当前费用结算与原生Position的数值权重差另外报告，不伪称原生浮点仓位与目标逐比特相同。每日按各自漂移持仓独立扣费，原始/1e-8有效超额分类并列，原收益和CI保持不变。

In [2]:
from src.workflow.benchmark_core_variance import run_benchmark_core_variance
summary,tables,conclusion=run_benchmark_core_variance(source_root=root,evidence_root=root,
    card_dir=card,recorder_uri=card/"qlib_recorder")
print({key:summary[key] for key in ("recorder_id","risk_parent_recorder_id","model_fits",
    "risk_calibration_fits","optimizer_solves","native_paths","native_all_pass","historical_gates","promotion_eligible")})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[76858:MainThread](2026-10-09 14:50:30,497) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[76858:MainThread](2026-10-09 14:50:30,497) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[76858:MainThread](2026-10-09 14:50:30,498) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/published_futu_affine_20260925_v1')}


[76858:MainThread](2026-10-09 14:50:30,527) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[76858:MainThread](2026-10-09 14:50:52,289) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/legacy_soxl_execution_20260925_v1')}


[76858:MainThread](2026-10-09 14:50:52,378) INFO - qlib.workflow - [exp.py:258] - Experiment 941399118967397861 starts running ...


[76858:MainThread](2026-10-09 14:50:52,382) INFO - qlib.workflow - [recorder.py:345] - Recorder 36c1620de6a3445eb7a1ff3e3a851dcc starts running under Experiment 941399118967397861 ...


[76858:MainThread](2026-10-09 14:50:57,269) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'recorder_id': '36c1620de6a3445eb7a1ff3e3a851dcc', 'risk_parent_recorder_id': '9a447c33dc934baeb1d4c3f2b9bf2122', 'model_fits': 0, 'risk_calibration_fits': 0, 'optimizer_solves': 0, 'native_paths': 10, 'native_all_pass': True, 'historical_gates': {'learned_core': False, 'close20_core': False}, 'promotion_eligible': False}


In [3]:
display(tables["development_input_reproduction.csv"])
display(tables["risk_mapping_summary.csv"])
display(tables["risk_realization_audit.csv"])
display(tables["native_audit.csv"][["strategy","cost_bps","sessions","status","maximum_session_return_error"]])
display(tables["backtest_benchmark_comparison.csv"][["strategy","cost_bps","window","relative_wealth","return_pct","max_drawdown_pct","cvar5_pct","turnover"]])
display(tables["variance_source_comparison.csv"][["candidate","reference","cost_bps","window","relative_wealth","log_excess_sum"]])
display(tables["gates.csv"])

,variance_key,repeat_id,rows,covariance_budget_support_provenance_exact,diagnostic_rebuilt_from_raw_calibrations,old_diagnostic_inputs_used,mean_objective_consumed
0,learned,1,1762,True,True,False,False
1,close20,1,1762,True,True,False,False


,strategy,variance_key,representation,sessions,maximum_budget_used_fraction,budget_underused_sessions,maximum_covariance_roundoff_adjustment,out_of_support_sessions,estimator_fallback_sessions,all_risk_targets_certified,mean_objective_used,optimizer_solves
0,learned_lever_cash,learned,lever_cash,2696,1.0,0,0.0,26,0,True,False,0
1,learned_core,learned,core,2696,1.0,0,0.0,26,0,True,False,0
2,close20_lever_cash,close20,lever_cash,2696,1.0,0,0.0,70,0,True,False,0
3,close20_core,close20,core,2696,1.0,0,0.0,70,0,True,False,0


,strategy,cost_bps,layer,sessions,maximum_target_weight_difference,maximum_realized_variance_budget_ratio,maximum_positive_variance_budget_excess,strict_float_variance_exceedance_rows,maximum_measured_perturbation_bound,within_measured_numerical_bound,target_or_economic_budget_modified
0,learned_lever_cash,10.0,independent_current_fee_settlement,2696,2.220446e-16,1.0,2.710505e-19,1232,5.114285e-18,True,False
1,learned_lever_cash,10.0,native_post_fee_open_positions,2696,2.220446e-16,1.0,3.252607e-19,1280,5.241994e-18,True,False
2,learned_core,10.0,independent_current_fee_settlement,2696,3.330669e-16,1.0,2.168404e-19,1527,5.111091e-18,True,False
3,learned_core,10.0,native_post_fee_open_positions,2696,5.551115e-16,1.0,3.252607e-19,1017,5.328158e-18,True,False
4,close20_lever_cash,10.0,independent_current_fee_settlement,2696,2.220446e-16,1.0,2.710505e-19,1268,5.196769e-18,True,False
5,close20_lever_cash,10.0,native_post_fee_open_positions,2696,2.220446e-16,1.0,3.252607e-19,1317,5.136327e-18,True,False
6,close20_core,10.0,independent_current_fee_settlement,2696,3.330669e-16,1.0,2.168404e-19,1557,5.089383e-18,True,False
7,close20_core,10.0,native_post_fee_open_positions,2696,4.579670e-16,1.0,2.710505e-19,1041,5.839161e-18,True,False
8,learned_lever_cash,25.0,independent_current_fee_settlement,2696,2.220446e-16,1.0,2.710505e-19,1235,5.108432e-18,True,False
9,learned_lever_cash,25.0,native_post_fee_open_positions,2696,2.220446e-16,1.0,3.794708e-19,1300,5.163370e-18,True,False


,strategy,cost_bps,sessions,status,maximum_session_return_error
0,soxx,10.0,2696,passed,4.440892e-16
1,learned_lever_cash,10.0,2696,passed,6.661338e-16
2,learned_core,10.0,2696,passed,4.440892e-16
3,close20_lever_cash,10.0,2696,passed,4.440892e-16
4,close20_core,10.0,2696,passed,4.440892e-16
5,soxx,25.0,2696,passed,2.220446e-16
6,learned_lever_cash,25.0,2696,passed,4.440892e-16
7,learned_core,25.0,2696,passed,6.661338e-16
8,close20_lever_cash,25.0,2696,passed,4.440892e-16
9,close20_core,25.0,2696,passed,4.440892e-16


,strategy,cost_bps,window,relative_wealth,return_pct,max_drawdown_pct,cvar5_pct,turnover
0,soxx,10.0,development,1.000000,325.747970,-47.373290,-4.416437,0.999001
1,soxx,10.0,reused_diagnostic,1.000000,398.621662,-42.789189,-5.295891,0.000000
2,learned_lever_cash,10.0,development,0.659913,180.956664,-36.265329,-3.462083,44.577613
3,learned_lever_cash,10.0,reused_diagnostic,0.601410,199.875947,-42.817743,-3.916175,22.479783
4,learned_core,10.0,development,0.668868,184.769187,-35.817510,-3.565545,133.225820
5,learned_core,10.0,reused_diagnostic,0.659032,228.607515,-41.397033,-3.979011,62.950785
6,close20_lever_cash,10.0,development,0.653757,178.335675,-34.898212,-3.632554,28.365959
7,close20_lever_cash,10.0,reused_diagnostic,0.762570,280.233732,-43.116547,-4.381080,15.853112
8,close20_core,10.0,development,0.701626,198.715867,-33.930835,-3.718788,66.981430
9,close20_core,10.0,reused_diagnostic,0.866219,331.915512,-41.023569,-4.452708,32.543575


,candidate,reference,cost_bps,window,relative_wealth,log_excess_sum
0,learned_core,close20_core,10.0,development,0.953311,-0.047814
1,learned_core,close20_core,10.0,reused_diagnostic,0.760814,-0.273366
2,learned_core,close20_core,25.0,development,0.863123,-0.147198
3,learned_core,close20_core,25.0,reused_diagnostic,0.726889,-0.318981


,cost_bps,aggregate_positive_net_log_excess,minimum_positive_signal_years,median_annual_log_excess_positive,leave_any_one_year_out_log_excess_positive,anchored_max_drawdown_not_worse_than_soxx,cvar5_not_worse_than_soxx,minimum_rolling_positive_excess_rate,stationary_block_20_lower_bound_positive,stationary_block_60_lower_bound_positive,...,raw_median_annual_log_excess_positive,raw_leave_any_one_year_out_log_excess_positive,raw_anchored_max_drawdown_not_worse_than_soxx,raw_cvar5_not_worse_than_soxx,raw_minimum_rolling_positive_excess_rate,raw_stationary_block_20_lower_bound_positive,raw_stationary_block_60_lower_bound_positive,raw_all_pass,raw_positive_years,strategy
0,10.0,False,False,False,False,True,True,False,False,False,...,False,False,True,True,False,False,False,False,1,learned_core
1,25.0,False,False,False,False,True,True,False,False,False,...,False,False,True,True,False,False,False,False,1,learned_core
2,10.0,False,False,False,False,True,True,False,False,False,...,False,False,True,True,False,False,False,False,3,close20_core
3,25.0,False,False,False,False,True,True,False,False,False,...,False,False,True,True,False,False,False,False,2,close20_core


## 结论与证据边界

小表入口：[summary.json](summary.json)、[development_input_reproduction.csv](development_input_reproduction.csv)、[risk_realization_audit.csv](risk_realization_audit.csv)、[gates.csv](gates.csv)。全部风险明细、目标、真实费用路径、原生Positions和报告保存在Qlib Recorder。

冻结价格仍未完成独立分配账本/总回报与历史PIT核验；后得到的分配信息不进入本批目标或调参。历史门即使通过也不等于已满足独立前向稳定超额。

In [4]:
display(Markdown(conclusion))

# 相同冻结条件风险下的基准核心与杠杆/现金表示

本批没有训练预测或重拟合风险校准：固定使用父研究repeat1与2016–2026全部年度参数，开发输入逐值复现；后段从原始风险表重建，没有冒用旧扩展波动诊断输入。

25bp完整结果如下：

- development/soxx：收益325.11%，相对SOXX财富1.000000，最大回撤-47.37%，CVaR5 -4.42%，换手0.9975。
- reused_diagnostic/soxx：收益398.62%，相对SOXX财富1.000000，最大回撤-42.79%，CVaR5 -5.30%，换手0.0000。
- development/learned_lever_cash：收益162.78%，相对SOXX财富0.618153，最大回撤-36.58%，CVaR5 -3.47%，换手44.5764。
- reused_diagnostic/learned_lever_cash：收益189.93%，相对SOXX财富0.581468，最大回撤-43.21%，CVaR5 -3.92%，换手22.4795。
- development/learned_core：收益133.18%，相对SOXX财富0.548519，最大回撤-36.84%，CVaR5 -3.58%，换手133.2153。
- reused_diagnostic/learned_core：收益199.00%，相对SOXX财富0.599644，最大回撤-42.58%，CVaR5 -3.99%，换手62.9468。
- development/close20_lever_cash：收益166.74%，相对SOXX财富0.627461，最大回撤-35.17%，CVaR5 -3.64%，换手28.3653。
- reused_diagnostic/close20_lever_cash：收益271.30%，相对SOXX财富0.744649，最大回撤-43.35%，CVaR5 -4.38%，换手15.8532。
- development/close20_core：收益170.16%，相对SOXX财富0.635505，最大回撤-34.40%，CVaR5 -3.73%，换手66.9732。
- reused_diagnostic/close20_core：收益311.34%，相对SOXX财富0.824946，最大回撤-41.53%，CVaR5 -4.46%，换手32.5419。

两个预登记候选的有效历史门：{'learned_core': False, 'close20_core': False}。不晋升，整体稳定超额目标继续。

相同风险源内core与lever_cash隔离持仓表示；learned_core与close20_core另作全成本比较。对SOXX的点估计优势不能自动证明学习波动预测有独立信息。

风险映射只使用过去可得协方差与冻结预算，既不读取均值也不调用凸优化器；协方差仅允许数值roundoff修复。目标风险逐行核验，预算因全仓上限未用满时如实登记。日交易按各自漂移持仓独立扣费，全部由原生Qlib重放。

年度/滚动与原始、1e-8有效正超额分类完整保留；支持域外标记不裁剪仓位。历史复用、旧风险标签精度、报价口径与16条次来源、连续份额、开盘成交及现金零收益仍有限制。下一步先审查信息增量、交易成本和尾部差异，再登记独立机制，不按本轮结果改预算或选风险源。

冻结价格不等于已用独立分配账本核验的总回报或完整历史PIT数据；任何历史通过仍需独立来源敏感性和真实分配账本验证。本批不混入后取得的分配信息，也不据此调参。

Qlib Recorder：36c1620de6a3445eb7a1ff3e3a851dcc；风险父Recorder：9a447c33dc934baeb1d4c3f2b9bf2122；as_of=2026-09-25。
